# 07 · Module 7: High-frequency order book

**Question.** Does the order-book signal survive the spread?

Present the model as an execution aid (when to cross the spread on NovaFin's own orders) rather than a stand-alone trading business.

**Data.** `nova_hft_orderbook.csv`: 120,000 rows, 10 stocks, 20 trading days (5–30 Jan 2026, day-first dates), minute-level timestamps. UP 42% / DOWN 42% / FLAT 16%. Split: whole days, 14 train / 3 validate / 3 test (`splits.day_block_split`).

**Professor's anchor.** S16 (spread, slippage, impact, fees; 0.20% gross to 0.05% net), S17 (imbalance and short-term momentum), S19 (deep learning against simpler models).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("07")

In [ ]:
hft = load("hft")
hft["Trading_Day"] = pd.to_datetime(hft["Trading_Day"], dayfirst=True)
LEAKAGE = ["Future_Return_100ms", "Future_Price_100ms"]   # backtest exit only, never a feature
hft["Price_Move_Class"] = hft["Price_Move_Class"].replace({"FLAT": "NO_MOVE"})
train, val, test = splits.day_block_split(hft["Trading_Day"])
hft.head()

> Evaluation standard: a naive baseline and a transparent model before any tree; PR-AUC against the base rate, Brier score and a calibration check for classifiers; the threshold chosen on validation data and reported on test data; SHAP or coefficients for the model that drives the decision; a stability check by period, segment or stock.

## 1. Features

OBI_Level1, OBI_3Level, Microprice_Minus_Mid, Relative_Spread, Total_Depth, depth by level, Order_Arrival_Rate, Cancellation_Rate, Short_Term_Volatility, Event_Type, Trade_Side, plus lagged changes within each stock. Nothing from `LEAKAGE`.

## 2. Models

Multinomial logistic regression on OBI alone (transparent baseline), then XGBoost on all features. An MLP or 1-D CNN only if time allows, kept only if it beats XGBoost on the test days.

## 3. Metrics

Macro-F1, per-class confusion matrix, log loss, calibration of P(UP) and P(DOWN).

## 4. Signal (Task 2)

Trade only when max(P(UP), P(DOWN)) exceeds a threshold. Enter at the far side (half the spread), exit at Future_Price_100ms (legitimate in the backtest only). Gross and net P&L per trade, hit rate, trades per day, break-even spread.

## 5. Execution use

For a large parent order: the saving from delaying when P(DOWN) is high and crossing early when P(UP) is high, against a naive immediate order.

## Handoff to Module 9

Net return per trade, daily P&L volatility, the capacity implied by depth at levels 1–3, and the operational risks (latency, model decay). Cap: at most 3% of the ₹1,000 cr.

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("07", lines)

## Traps to check before handing over

Minute-level timestamps: any feature built from other rows in the same minute can leak. A positive gross P&L means little until it is shown net of spread.